# JSON in SQLite

## Quick notes
### What this covers
- SQLite's built-in JSON functions: reading (`json_extract`, `->`, `->>`), writing (`json_set`, `json_insert`, `json_replace`, `json_remove`, `json_patch`), building (`json_object`, `json_array`, `json_group_array`, `json_group_object`) and exploding (`json_each`, `json_tree`).
- Storing documents in a column, querying them, joining on them, validating them and indexing them.
- When JSON is the right tool and when plain tables are better.
### Key points
- SQLite has no JSON type. JSON is just TEXT that the `json_*` functions parse every time.
- `->` returns JSON text (strings keep their quotes). `->>` and `json_extract` return a plain SQL value (TEXT, INTEGER, REAL or NULL).
- `json_each` turns an array (or object) into rows. `json_tree` walks the whole nested document.
- `CHECK (json_valid(col))` stops broken JSON from being stored.
- You can index a JSON field with an expression index or a generated column.
### Gotchas (the part worth revising twice)
- A missing path gives SQL `NULL`. A JSON `null` value also gives SQL `NULL` through `->>`. Use `json_type` to tell them apart.
- `json_extract` of an object or array returns JSON text, but `json_extract` of a string returns the bare string. `->` always returns JSON, so a string comes back quoted.
- JSON booleans become the integers 1 and 0. JSON numbers can be INTEGER or REAL depending on how they were written.
- Bad JSON raises "malformed JSON" only when a function reads it. Without a CHECK, junk is stored happily.
- To use an index the query must repeat the exact same expression that was indexed.
- Fast lookups inside arrays need `json_each` and are not indexable. Many-to-many data belongs in real tables.
- `json_valid()` and the JSON functions need a build with JSON support. It is on by default since 3.38; older or unusual builds may differ.
### When it matters
- Semi-structured payloads: API responses, event logs, user settings, product attributes that differ per row.
- Keeping a raw copy of an incoming document next to a few relational columns you actually query.
### One-line recap example
- `SELECT doc ->> '$.user.name' FROM events WHERE doc ->> '$.type' = 'click';` means "reach into the JSON text and return the user name for click events".

## 01. Seed data: a JSON column
- `events` stores one JSON document per row, with a `CHECK` so only valid JSON gets in.
- `customers` is a normal relational table we will join to later.
- Dates are fixed text values.

In [1]:
# setup: an in-memory SQLite database and a %%sql cell magic that prints result tables
import sqlite3
from IPython.core.magic import register_cell_magic

con = sqlite3.connect(":memory:", isolation_level=None)   # autocommit, so BEGIN/COMMIT work as written
con.execute("PRAGMA foreign_keys = ON")

def _table(cols, rows):
    cells = [[("NULL" if v is None else str(v)) for v in r] for r in rows]
    widths = [max([len(c)] + [len(r[i]) for r in cells]) for i, c in enumerate(cols)]
    line = lambda vals: " | ".join(v.ljust(w) for v, w in zip(vals, widths))
    print(line(cols)); print("-+-".join("-" * w for w in widths))
    for r in cells: print(line(r))
    print(f"({len(rows)} row{'s' if len(rows) != 1 else ''})\n")

def _statements(text):
    """Split a cell into statements (handles several per line, triggers, quoted semicolons)."""
    out, start = [], 0
    for i, ch in enumerate(text):
        if ch == ";" and sqlite3.complete_statement(text[start:i + 1]):
            out.append(text[start:i + 1].strip()); start = i + 1
    if text[start:].strip():
        out.append(text[start:].strip())
    return out

@register_cell_magic
def sql(line, cell):
    """Run every statement in the cell; print rows for statements that return them."""
    expect_error = "expect_error" in line
    for stmt in _statements(cell):
        try:
            cur = con.execute(stmt)
            if cur.description:
                _table([d[0] for d in cur.description], cur.fetchall())
        except sqlite3.Error as exc:
            if not expect_error:
                raise
            print(f"ERROR: {exc}\n")

In [2]:
%%sql
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    name        TEXT NOT NULL,
    city        TEXT NOT NULL
);
INSERT INTO customers VALUES (1,'Asha','Pune'),(2,'Ben','Leeds'),(3,'Chloe','Oslo'),(4,'Dev','Pune');

CREATE TABLE events (
    event_id INTEGER PRIMARY KEY,
    doc      TEXT NOT NULL CHECK (json_valid(doc))
);
INSERT INTO events VALUES
 (1, '{"type":"click","user":{"id":1,"name":"Asha"},"page":"/home","tags":["promo","mobile"],"at":"2025-03-01T09:00:00"}'),
 (2, '{"type":"purchase","user":{"id":2,"name":"Ben"},"items":[{"sku":"A1","qty":2,"price":9.5},{"sku":"B7","qty":1,"price":20}],"coupon":null,"at":"2025-03-01T10:30:00"}'),
 (3, '{"type":"click","user":{"id":1,"name":"Asha"},"page":"/pricing","tags":[],"at":"2025-03-02T08:15:00"}'),
 (4, '{"type":"purchase","user":{"id":3,"name":"Chloe"},"items":[{"sku":"A1","qty":1,"price":9.5}],"coupon":"SPRING","paid":true,"at":"2025-03-02T12:00:00"}'),
 (5, '{"type":"click","user":{"id":4,"name":"Dev"},"page":"/home","tags":["email"],"at":"2025-03-03T17:45:00"}');
SELECT count(*) AS events_loaded FROM events;

events_loaded
-------------
5            
(1 row)



## 02. json_valid, json() and the CHECK constraint
- `json_valid(x)` returns 1 or 0. `json(x)` parses and returns a compact minified copy, and raises an error if the text is bad.
- The `CHECK` in the seed makes bad JSON impossible to insert.
- Valid JSON must use double quotes and no trailing commas.

In [3]:
%%sql
SELECT json_valid('{"a":1}')   AS good,
       json_valid('{a:1}')     AS unquoted_key,
       json_valid('{"a":1,}')  AS trailing_comma,
       json_valid('{''a'':1}') AS single_quotes,
       json('{ "a" :  [1, 2,   3] }') AS minified;

good | unquoted_key | trailing_comma | single_quotes | minified     
-----+--------------+----------------+---------------+--------------
1    | 0            | 0              | 0             | {"a":[1,2,3]}
(1 row)



In [4]:
%%sql expect_error
INSERT INTO events VALUES (99, '{"type": broken');

ERROR: CHECK constraint failed: json_valid(doc)



In [5]:
%%sql
SELECT count(*) AS still_five FROM events;

still_five
----------
5         
(1 row)



## 03. Reading values: json_extract, -> and ->>
- Paths start with `$`. Use `.key` for object keys and `[n]` for array positions (0-based, `[#-1]` is the last).
- `->>` and `json_extract` give a SQL value. `->` gives JSON text, so a string keeps its double quotes.
- `->` and `->>` also accept a bare key like `'type'` instead of `'$.type'` (3.38+).

In [6]:
%%sql
SELECT event_id,
       json_extract(doc, '$.type')      AS extract_text,
       doc ->> '$.type'                 AS arrow2_text,
       doc -> '$.type'                  AS arrow1_json,
       doc ->> 'type'                   AS bare_key,
       doc ->> '$.user.name'            AS user_name,
       doc ->> '$.tags[0]'              AS first_tag,
       doc ->> '$.tags[#-1]'            AS last_tag
FROM events
ORDER BY event_id;

event_id | extract_text | arrow2_text | arrow1_json | bare_key | user_name | first_tag | last_tag
---------+--------------+-------------+-------------+----------+-----------+-----------+---------
1        | click        | click       | "click"     | click    | Asha      | promo     | mobile  
2        | purchase     | purchase    | "purchase"  | purchase | Ben       | NULL      | NULL    
3        | click        | click       | "click"     | click    | Asha      | NULL      | NULL    
4        | purchase     | purchase    | "purchase"  | purchase | Chloe     | NULL      | NULL    
5        | click        | click       | "click"     | click    | Dev       | email     | email   
(5 rows)



## 04. Objects and arrays come back as JSON text
- When the path points to an object or array, every reader returns JSON text.
- You can chain: `doc -> 'user' ->> 'id'` is the same as `doc ->> '$.user.id'`.
- `json_array_length` and `json_type` inspect what is at a path.

In [7]:
%%sql
SELECT event_id,
       doc -> '$.user'                        AS user_json,
       doc -> 'user' ->> 'id'                 AS chained_id,
       json_array_length(doc, '$.tags')       AS tag_count,
       json_type(doc, '$.tags')               AS tags_type,
       json_type(doc, '$.user')               AS user_type
FROM events
ORDER BY event_id;

event_id | user_json               | chained_id | tag_count | tags_type | user_type
---------+-------------------------+------------+-----------+-----------+----------
1        | {"id":1,"name":"Asha"}  | 1          | 2         | array     | object   
2        | {"id":2,"name":"Ben"}   | 2          | NULL      | NULL      | object   
3        | {"id":1,"name":"Asha"}  | 1          | 0         | array     | object   
4        | {"id":3,"name":"Chloe"} | 3          | NULL      | NULL      | object   
5        | {"id":4,"name":"Dev"}   | 4          | 1         | array     | object   
(5 rows)



## 05. NULL versus 'null' versus missing
- Event 2 has `"coupon": null` (JSON null). Event 1 has no `coupon` key at all. Event 4 has a real coupon.
- `->>` returns SQL `NULL` for both missing and JSON null, so `IS NULL` cannot tell them apart.
- `json_type` can: it gives `'null'` (text) for JSON null and SQL `NULL` for a missing path.
- `->` on a JSON null returns the text `null`, which is not SQL NULL.

In [8]:
%%sql
SELECT event_id,
       doc ->> '$.coupon'            AS coupon_value,
       doc -> '$.coupon'             AS coupon_json,
       json_type(doc, '$.coupon')    AS coupon_type,
       doc ->> '$.coupon' IS NULL    AS looks_null,
       (doc -> '$.coupon') IS NULL   AS arrow_is_null
FROM events
WHERE event_id IN (1, 2, 4)
ORDER BY event_id;

event_id | coupon_value | coupon_json | coupon_type | looks_null | arrow_is_null
---------+--------------+-------------+-------------+------------+--------------
1        | NULL         | NULL        | NULL        | 1          | 1            
2        | NULL         | null        | null        | 1          | 0            
4        | SPRING       | "SPRING"    | text        | 0          | 0            
(3 rows)



## 06. Numbers, booleans and text pitfalls
- JSON `true` and `false` come out as `1` and `0` with `->>`. With `->` you get the text `true` or `false`.
- A number inside quotes is TEXT in JSON and stays text in SQL. Compare it as text or cast it.
- Integers stay INTEGER, decimals become REAL. Check with `typeof()`.

In [9]:
%%sql
SELECT json_extract('{"ok":true,"off":false}', '$.ok')   AS ok_as_sql,
       '{"ok":true}' ->> '$.ok'                          AS ok_arrow2,
       '{"ok":true}' ->  '$.ok'                          AS ok_arrow1,
       typeof('{"n":7}' ->> '$.n')                       AS int_type,
       typeof('{"n":7.5}' ->> '$.n')                     AS real_type,
       typeof('{"n":"7"}' ->> '$.n')                     AS quoted_number_type,
       json_type('{"n":"7"}', '$.n')                     AS quoted_json_type;

ok_as_sql | ok_arrow2 | ok_arrow1 | int_type | real_type | quoted_number_type | quoted_json_type
----------+-----------+-----------+----------+-----------+--------------------+-----------------
1         | 1         | true      | integer  | real      | text               | text            
(1 row)



In [10]:
%%sql
SELECT event_id, doc ->> '$.paid' AS paid_raw, json_type(doc, '$.paid') AS paid_type
FROM events
WHERE event_id IN (2, 4)
ORDER BY event_id;

event_id | paid_raw | paid_type
---------+----------+----------
2        | NULL     | NULL     
4        | 1        | true     
(2 rows)



## 07. Filtering rows by JSON content
- Put the `->>` expression in `WHERE` like any column.
- To test existence of a key, compare `json_type(...) IS NOT NULL`.
- Comparing a `->` result to a string needs the quotes: `'"click"'`. That is why `->>` is usually what you want in a filter.

In [11]:
%%sql
SELECT event_id, doc ->> '$.page' AS page
FROM events
WHERE doc ->> '$.type' = 'click' AND doc ->> '$.page' = '/home'
ORDER BY event_id;

event_id | page 
---------+------
1        | /home
5        | /home
(2 rows)



In [12]:
%%sql
SELECT 'arrow1_without_quotes' AS test, count(*) AS rows_found FROM events WHERE doc -> '$.type' = 'click'
UNION ALL
SELECT 'arrow1_with_quotes',            count(*) FROM events WHERE doc -> '$.type' = '"click"'
UNION ALL
SELECT 'has_coupon_key',                count(*) FROM events WHERE json_type(doc, '$.coupon') IS NOT NULL
ORDER BY test;

test                  | rows_found
----------------------+-----------
arrow1_with_quotes    | 3         
arrow1_without_quotes | 0         
has_coupon_key        | 2         
(3 rows)



## 08. Exploding arrays with json_each
- `json_each(json, path)` is a table-valued function: one row per array element (or object member).
- Columns: `key`, `value`, `type`, `atom`, `id`, `parent`, `fullkey`, `path`.
- Join it to the row that owns the array with a comma join (a lateral-style join that SQLite allows for these functions).
- Tags: events with an empty array or no array give no rows at all.

In [13]:
%%sql
SELECT e.event_id, t.key AS position, t.value AS tag
FROM events e, json_each(e.doc, '$.tags') AS t
ORDER BY e.event_id, t.key;

event_id | position | tag   
---------+----------+-------
1        | 0        | promo 
1        | 1        | mobile
5        | 0        | email 
(3 rows)



In [14]:
%%sql
SELECT t.value AS tag, count(*) AS uses
FROM events e, json_each(e.doc, '$.tags') AS t
GROUP BY t.value
ORDER BY uses DESC, tag;

tag    | uses
-------+-----
email  | 1   
mobile | 1   
promo  | 1   
(3 rows)



## 09. Arrays of objects: line items
- Each purchase has an `items` array of objects. Explode it, then read fields from each element with `->>`.
- Now normal SQL works: totals, group by sku, joins.
- Revenue is `qty * price` per element.

In [15]:
%%sql
SELECT e.event_id,
       i.value ->> '$.sku'                              AS sku,
       i.value ->> '$.qty'                              AS qty,
       i.value ->> '$.price'                            AS price,
       (i.value ->> '$.qty') * (i.value ->> '$.price')  AS line_total
FROM events e, json_each(e.doc, '$.items') AS i
WHERE e.doc ->> '$.type' = 'purchase'
ORDER BY e.event_id, i.key;

event_id | sku | qty | price | line_total
---------+-----+-----+-------+-----------
2        | A1  | 2   | 9.5   | 19.0      
2        | B7  | 1   | 20    | 20        
4        | A1  | 1   | 9.5   | 9.5       
(3 rows)



In [16]:
%%sql
SELECT i.value ->> '$.sku' AS sku,
       sum(i.value ->> '$.qty')                              AS units,
       sum((i.value ->> '$.qty') * (i.value ->> '$.price'))  AS revenue
FROM events e, json_each(e.doc, '$.items') AS i
GROUP BY sku
ORDER BY revenue DESC;

sku | units | revenue
----+-------+--------
A1  | 3     | 28.5   
B7  | 1     | 20     
(2 rows)



## 10. Joining JSON to relational tables
- Pull the foreign key out of the JSON with `->>` and join like any other column.
- Types matter: `->>` on a JSON integer gives an INTEGER, so it matches `customer_id` directly.
- Use `LEFT JOIN` instead if you must keep events whose user is missing from `customers` (all four users exist here, so the result is the same).

In [17]:
%%sql
SELECT c.name, c.city, count(*) AS events_count,
       sum(e.doc ->> '$.type' = 'purchase') AS purchases
FROM events e
JOIN customers c ON c.customer_id = e.doc ->> '$.user.id'
GROUP BY c.customer_id
ORDER BY events_count DESC, c.name;

name  | city  | events_count | purchases
------+-------+--------------+----------
Asha  | Pune  | 2            | 0        
Ben   | Leeds | 1            | 1        
Chloe | Oslo  | 1            | 1        
Dev   | Pune  | 1            | 0        
(4 rows)



## 11. Walking a nested document with json_tree
- `json_tree` visits every node at every depth, not only direct children like `json_each`.
- `fullkey` is the exact path to the node. `atom` holds the value of leaves (NULL for objects and arrays).
- Handy to find where a value lives in an unknown document.

In [18]:
%%sql
SELECT fullkey, type, atom
FROM json_tree('{"user":{"id":2,"name":"Ben"},"items":[{"sku":"A1","qty":2}],"coupon":null}')
WHERE type NOT IN ('object', 'array')
ORDER BY id;

fullkey        | type    | atom
---------------+---------+-----
$.user.id      | integer | 2   
$.user.name    | text    | Ben 
$.items[0].sku | text    | A1  
$.items[0].qty | integer | 2   
$.coupon       | null    | NULL
(5 rows)



In [19]:
%%sql
SELECT e.event_id, t.fullkey
FROM events e, json_tree(e.doc) AS t
WHERE t.atom = 'A1'
ORDER BY e.event_id, t.fullkey;

event_id | fullkey       
---------+---------------
2        | $.items[0].sku
4        | $.items[0].sku
(2 rows)



## 12. Changing documents: json_set, json_insert, json_replace, json_remove
- `json_set` creates or overwrites. `json_insert` only creates (never overwrites). `json_replace` only overwrites (never creates).
- `json_remove` deletes a path. All of them return a new JSON text, so use `UPDATE ... SET doc = json_set(...)`.
- Setting a text value with `json_set` stores a JSON string; to insert a ready-made JSON fragment, wrap it in `json()`.

In [20]:
%%sql
SELECT json_set('{"a":1}',     '$.a', 10, '$.b', 20) AS set_both,
       json_insert('{"a":1}',  '$.a', 10, '$.b', 20) AS insert_only_new,
       json_replace('{"a":1}', '$.a', 10, '$.b', 20) AS replace_only_existing,
       json_remove('{"a":1,"b":2}', '$.a')           AS removed;

set_both        | insert_only_new | replace_only_existing | removed
----------------+-----------------+-----------------------+--------
{"a":10,"b":20} | {"a":1,"b":20}  | {"a":10}              | {"b":2}
(1 row)



In [21]:
%%sql
SELECT json_set('{"a":1}', '$.obj', '{"x":1}')       AS fragment_as_string,
       json_set('{"a":1}', '$.obj', json('{"x":1}')) AS fragment_as_json;

fragment_as_string        | fragment_as_json     
--------------------------+----------------------
{"a":1,"obj":"{\"x\":1}"} | {"a":1,"obj":{"x":1}}
(1 row)



## 13. Updating stored rows and arrays
- Add a field to every click event, append to an array with the `[#]` path, and remove an element by position.
- `json_insert(doc, '$.tags[#]', 'x')` appends to the end of an array.
- Each update rewrites the whole document text, so very large documents are costly to edit often.

In [22]:
%%sql
UPDATE events SET doc = json_set(doc, '$.source', 'web') WHERE doc ->> '$.type' = 'click';
UPDATE events SET doc = json_insert(doc, '$.tags[#]', 'seen') WHERE event_id = 1;
UPDATE events SET doc = json_remove(doc, '$.tags[0]')         WHERE event_id = 1;
SELECT event_id, doc -> '$.tags' AS tags, doc ->> '$.source' AS source
FROM events
WHERE doc ->> '$.type' = 'click'
ORDER BY event_id;

event_id | tags              | source
---------+-------------------+-------
1        | ["mobile","seen"] | web   
3        | []                | web   
5        | ["email"]         | web   
(3 rows)



## 14. json_patch: merge documents
- `json_patch(target, patch)` applies a JSON merge patch (RFC 7396). Matching keys are overwritten, new keys are added, a `null` in the patch deletes the key.
- Arrays are replaced whole, not merged element by element.
- Nice for "apply these settings changes" operations.

In [23]:
%%sql
SELECT json_patch('{"theme":"light","font":12,"beta":true}',
                  '{"theme":"dark","beta":null,"lang":"en"}') AS merged,
       json_patch('{"tags":["a","b"]}', '{"tags":["c"]}')     AS arrays_replaced;

merged                                 | arrays_replaced
---------------------------------------+----------------
{"theme":"dark","font":12,"lang":"en"} | {"tags":["c"]} 
(1 row)



## 15. Building JSON from tables
- `json_object(k, v, ...)` and `json_array(...)` build values. Text values become JSON strings; numbers stay numbers.
- Aggregates: `json_group_array(x)` collects rows into an array, `json_group_object(k, v)` into an object.
- A value produced by another `json_*` function is treated as JSON and nested properly (see the city/people example). Plain text that merely looks like JSON would be stored as an escaped string; wrap it in `json()` first.

In [24]:
%%sql
SELECT json_object('name', name, 'city', city) AS customer_doc
FROM customers
ORDER BY customer_id;

customer_doc                  
------------------------------
{"name":"Asha","city":"Pune"} 
{"name":"Ben","city":"Leeds"} 
{"name":"Chloe","city":"Oslo"}
{"name":"Dev","city":"Pune"}  
(4 rows)



In [25]:
%%sql
SELECT city,
       json_group_array(name ORDER BY name) AS names,
       count(*)                              AS n
FROM customers
GROUP BY city
ORDER BY city;

city  | names          | n
------+----------------+--
Leeds | ["Ben"]        | 1
Oslo  | ["Chloe"]      | 1
Pune  | ["Asha","Dev"] | 2
(3 rows)



In [26]:
%%sql
SELECT json_group_object(name, city) AS name_to_city
FROM (SELECT name, city FROM customers ORDER BY customer_id);

name_to_city                                             
---------------------------------------------------------
{"Asha":"Pune","Ben":"Leeds","Chloe":"Oslo","Dev":"Pune"}
(1 row)



In [27]:
%%sql
SELECT json_object('city', city,
                   'people', (SELECT json_group_array(c2.name) FROM (SELECT name FROM customers c2 WHERE c2.city = c.city ORDER BY name) AS c2)
       ) AS city_doc
FROM (SELECT DISTINCT city FROM customers) AS c
ORDER BY city;

city_doc                               
---------------------------------------
{"city":"Leeds","people":["Ben"]}      
{"city":"Oslo","people":["Chloe"]}     
{"city":"Pune","people":["Asha","Dev"]}
(3 rows)



## 16. Aggregating events stored as JSON
- Daily activity: take the date part of an ISO timestamp with `substr(...)` or `date(...)`, then group.
- Conditional aggregation with `FILTER` works on JSON-derived values like any other.
- See 01-Basics/05-Dates-And-Times for date handling and 02-Querying/02-Aggregation-And-Grouping for `FILTER`.

In [28]:
%%sql
SELECT date(doc ->> '$.at')                                   AS day,
       count(*)                                               AS events_count,
       count(*) FILTER (WHERE doc ->> '$.type' = 'click')     AS clicks,
       count(*) FILTER (WHERE doc ->> '$.type' = 'purchase')  AS purchases
FROM events
GROUP BY day
ORDER BY day;

day        | events_count | clicks | purchases
-----------+--------------+--------+----------
2025-03-01 | 2            | 1      | 1        
2025-03-02 | 2            | 1      | 1        
2025-03-03 | 1            | 1      | 0        
(3 rows)



## 17. Indexing JSON: expression index
- A normal index cannot see inside JSON text. Create an index on the expression itself.
- The query must use the identical expression for the planner to use the index. `->>` and `json_extract` look equivalent, but the second plan below shows a full scan, so the index is not used when you switch spelling.
- Use `EXPLAIN QUERY PLAN` to check (see 07-Performance/01-Indexes-And-Query-Plans for reading plans).

In [29]:
%%sql
CREATE INDEX idx_events_type ON events ((doc ->> '$.type'));
EXPLAIN QUERY PLAN SELECT event_id FROM events WHERE doc ->> '$.type' = 'purchase';

id | parent | notused | detail                                              
---+--------+---------+-----------------------------------------------------
3  | 0      | 0       | SEARCH events USING INDEX idx_events_type (<expr>=?)
(1 row)



In [30]:
%%sql
EXPLAIN QUERY PLAN SELECT event_id FROM events WHERE json_extract(doc, '$.type') = 'purchase';

id | parent | notused | detail     
---+--------+---------+------------
2  | 0      | 0       | SCAN events
(1 row)



## 18. Indexing JSON: generated column
- A generated column holds a JSON field as a real column. It reads like a normal column and can be indexed.
- `VIRTUAL` computes on read, `STORED` saves the value. Virtual columns can be added with `ALTER TABLE ... ADD COLUMN`; stored ones cannot.
- The queries become simpler (`WHERE user_id = 1`) and the plan uses the index.

In [31]:
%%sql
ALTER TABLE events ADD COLUMN user_id INTEGER GENERATED ALWAYS AS (doc ->> '$.user.id') VIRTUAL;
CREATE INDEX idx_events_user ON events (user_id);
SELECT event_id, user_id FROM events WHERE user_id = 1 ORDER BY event_id;
EXPLAIN QUERY PLAN SELECT event_id FROM events WHERE user_id = 1;

event_id | user_id
---------+--------
1        | 1      
3        | 1      
(2 rows)

id | parent | notused | detail                                               
---+--------+---------+------------------------------------------------------
3  | 0      | 0       | SEARCH events USING INDEX idx_events_user (user_id=?)
(1 row)



## 19. JSON versus relational: the trade-offs
- JSON wins for flexible, rarely-queried, per-row-different attributes, and for storing a raw payload as received.
- Plain tables win when you filter, join, sum or enforce rules (types, foreign keys, `NOT NULL`, `UNIQUE`).
- Middle path: keep the document, and lift the 2-3 fields you query into generated columns.
- Below: the same purchase items as a real table. Constraints now do real work.

In [32]:
%%sql
CREATE TABLE purchase_items (
    event_id INTEGER NOT NULL REFERENCES events(event_id),
    sku      TEXT    NOT NULL,
    qty      INTEGER NOT NULL CHECK (qty > 0),
    price    REAL    NOT NULL CHECK (price >= 0),
    PRIMARY KEY (event_id, sku)
);
INSERT INTO purchase_items
SELECT e.event_id, i.value ->> '$.sku', i.value ->> '$.qty', i.value ->> '$.price'
FROM events e, json_each(e.doc, '$.items') AS i
ORDER BY e.event_id, i.key;
SELECT * FROM purchase_items ORDER BY event_id, sku;

event_id | sku | qty | price
---------+-----+-----+------
2        | A1  | 2   | 9.5  
2        | B7  | 1   | 20.0 
4        | A1  | 1   | 9.5  
(3 rows)



In [33]:
%%sql expect_error
INSERT INTO purchase_items VALUES (2, 'Z9', -1, 5.0);

ERROR: CHECK constraint failed: qty > 0



## 20. JSONB note (3.45+)
- SQLite 3.45 added `jsonb(...)`, a binary form of JSON. Functions accept either form and it avoids re-parsing text.
- Convert with `jsonb(x)` and back with `json(x)`. A JSONB value is a BLOB (see `stored_type`), so convert it back with `json()` to display it.
- `json_valid(x)` with its default flag checks for text JSON, so a JSONB blob gives 0. Flag `8` checks for a well-formed JSONB blob, which gives 1.
- Stored JSONB is only readable by SQLite 3.45+. Keep plain text JSON if other tools or older builds read the file.

In [34]:
%%sql
SELECT typeof(jsonb('{"a":[1,2]}'))              AS stored_type,
       json(jsonb('{"a":[1,2]}'))                AS back_to_text,
       jsonb('{"a":[1,2]}') ->> '$.a[1]'         AS read_from_jsonb,
       json_valid(jsonb('{"a":1}'))              AS valid_as_text,
       json_valid(jsonb('{"a":1}'), 8)           AS valid_as_jsonb;

stored_type | back_to_text | read_from_jsonb | valid_as_text | valid_as_jsonb
------------+--------------+-----------------+---------------+---------------
blob        | {"a":[1,2]}  | 2               | 0             | 1             
(1 row)

